In [2]:
# =====================================================
# 1. IMPORT LIBRARIES
# =====================================================

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/krupalpatel07/goldman-sachs-dataset/GS.csv


In [3]:

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

plt.rcParams['figure.figsize'] = (12,6)

In [15]:
import plotly.io as pio

pio.renderers.default = 'iframe'

In [16]:
# =====================================================
# 2. LOAD DATA
# =====================================================
file_path = "/kaggle/input/datasets/krupalpatel07/goldman-sachs-dataset/GS.csv"
df = pd.read_csv(file_path)

In [17]:
# =====================================================
# 3. PREPROCESSING
# =====================================================
df.columns = [c.lower() for c in df.columns]
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')
df.set_index('date', inplace=True)

In [18]:
# =====================================================
# 4. STORY HEADER STYLE
# =====================================================
from IPython.display import display, HTML

def news_block(title, subtitle):
    display(HTML(f"""
    <div style="
        background:#111;
        padding:20px;
        border-left:8px solid #f5c542;
        margin-top:20px;
    ">
        <h1 style="color:#f5c542;">{title}</h1>
        <p style="color:white; font-size:16px;">{subtitle}</p>
    </div>
    """))

news_block("📊 Market Narrative","Tracking structural behavior of Goldman Sachs price action")

In [19]:
# =====================================================
# 5. TREND STORY VISUAL
# =====================================================
fig = px.line(df, y='close', title="Closing Price Storyline")
fig.show()

In [20]:
# =====================================================
# 6. RETURN REGIMES
# =====================================================
news_block("⚡ Return Regimes","Identifying bullish and bearish phases")

df['returns'] = df['close'].pct_change()

df['regime'] = np.where(df['returns'] > 0, 'Bull', 'Bear')

fig = px.scatter(df, x=df.index, y='returns', color='regime')
fig.show()

In [21]:
# =====================================================
# 7. DRAWdown ANALYSIS
# =====================================================
news_block("📉 Drawdown Risk","Understanding capital erosion phases")

cum_max = df['close'].cummax()
df['drawdown'] = (df['close'] - cum_max) / cum_max

fig = px.area(df, y='drawdown', title="Drawdown Curve")
fig.show()

In [22]:
# =====================================================
# 8. VOLUME SHOCK DETECTION
# =====================================================
news_block("📦 Volume Shock Detector","Unusual trading activity signals")

vol_mean = df['volume'].rolling(20).mean()
vol_std = df['volume'].rolling(20).std()

df['volume_spike'] = df['volume'] > (vol_mean + 2*vol_std)

fig = px.scatter(df, x=df.index, y='volume', color='volume_spike')
fig.show()


In [23]:
# =====================================================
# 9. PCA FEATURE REDUCTION
# =====================================================
news_block("🧠 Dimensionality Compression","Extracting latent structure")

features = df[['open','high','low','close','volume']].dropna()
scaler = StandardScaler()
scaled = scaler.fit_transform(features)

pca = PCA(n_components=2)
pca_features = pca.fit_transform(scaled)

pca_df = pd.DataFrame(pca_features, columns=['PC1','PC2'])

fig = px.scatter(pca_df, x='PC1', y='PC2')
fig.show()

In [24]:
# =====================================================
# 10. TIME STRUCTURE (SEASONALITY)
# =====================================================
news_block("📆 Time Structure","Seasonality & calendar effects")

df['month'] = df.index.month
monthly_returns = df.groupby('month')['returns'].mean()

fig = px.bar(monthly_returns, title="Average Monthly Returns")
fig.show()

In [25]:
# =====================================================
# 11. FINAL STORY
# =====================================================
news_block("📝 Final Story","Key takeaways from market behavior")

print("""
1. Alternating bull-bear regimes define structure.
2. Drawdowns highlight risk periods.
3. Volume spikes often precede large moves.
4. PCA reveals underlying latent dynamics.
5. Seasonal patterns can be exploited.
""")

# =====================================================
# END
# =====================================================



1. Alternating bull-bear regimes define structure.
2. Drawdowns highlight risk periods.
3. Volume spikes often precede large moves.
4. PCA reveals underlying latent dynamics.
5. Seasonal patterns can be exploited.

